## DATA INGESTION

In [2]:
import os
from pathlib import Path
import magic
import fitz
import pandas as pd
from tqdm import tqdm

In [7]:
raw_data_dir = Path("../data/raw")
raw_data_dir.mkdir(parents=True, exist_ok=True)


In [9]:
def is_pdf(file_path):
    try:
        file_type = magic.from_file(str(file_path), mime=True)
        print(f"Detected MIME for {file_path}: {file_type}")
        return file_type == "application/pdf"
    except Exception as e:
        print("Error:",e)
        return False

In [10]:
is_pdf("../data/raw/sample.pdf")

Detected MIME for ../data/raw/sample.pdf: application/pdf


True

In [14]:
def extract_pdf_metadata(pdf_path):
    pdf_path = Path(pdf_path)
    try:
        doc = fitz.open(pdf_path)
        meta = doc.metadata
        info = {
            "file_name": pdf_path.name,
            "path": str(pdf_path),
            "pages": len(doc),
            "title": meta.get("title", None),
            "author": meta.get("author", None),
            "filesize_kb": round(os.path.getsize(pdf_path) / 1024, 2),
        }
        doc.close()
        return info
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
        return None

In [16]:
result = extract_pdf_metadata("../data/raw/sample.pdf")
result

{'file_name': 'sample.pdf',
 'path': '../data/raw/sample.pdf',
 'pages': 19,
 'title': 'AutoFactory Dataset to Support AI in Manufacturing Systems',
 'author': 'Abderrahmane Boudribila',
 'filesize_kb': 2327.14}

In [26]:
pdf_files = list(raw_data_dir.glob("*.pdf"))
pdf_files

[PosixPath('../data/raw/sample4.pdf'),
 PosixPath('../data/raw/sample5.pdf'),
 PosixPath('../data/raw/sample6.pdf'),
 PosixPath('../data/raw/sample2.pdf'),
 PosixPath('../data/raw/sample.pdf'),
 PosixPath('../data/raw/sample3.pdf')]

In [27]:
metadata_list = []

for pdf in tqdm(pdf_files):
    meta = extract_pdf_metadata(pdf)
    if meta:
        metadata_list.append(meta)

df_metadata = pd.DataFrame(metadata_list)
df_metadata
        
    


100%|██████████| 6/6 [00:00<00:00, 96.13it/s]


,file_name,path,pages,title,author,filesize_kb
0,sample4.pdf,../data/raw/sample4.pdf,10,,,212.08
1,sample5.pdf,../data/raw/sample5.pdf,10,,,559.81
2,sample6.pdf,../data/raw/sample6.pdf,125,Microsoft PowerPoint - RR-FY 2025-26 (H1),RAVINDRA KUMAR,7202.36
3,sample2.pdf,../data/raw/sample2.pdf,324,,,20366.17
4,sample.pdf,../data/raw/sample.pdf,19,AutoFactory Dataset to Support AI in Manufactu...,Abderrahmane Boudribila,2327.14
5,sample3.pdf,../data/raw/sample3.pdf,13,,,228.86


In [28]:
df_metadata.to_csv("../data/raw/metadata.csv", index=False)
print("saved metadata.csv")

saved metadata.csv


Pixmap(DeviceRGB, (0, 0, 423, 639), 0)